<a href="https://colab.research.google.com/github/jimmynewland/colabnotebooks/blob/main/Using_AI_to_Visualize_and_Analyze_Human_Pulse_Signal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using AI to Visualize and Analyze Human Pulse Signal


We will be visualizing and analyzing pulse signal data. The actual output of the analysis is often called a photoplethysmogram (ppg). This technique is more generally known as photoplethysmography, which is a fun but challenging name to say aloud.

AI tools can help build understanding
* The [remote PPG](https://thinkingwithcode.com/ppg/) (rPPG) algorithm uses an AI tool called [Haar cascades](https://thinkingwithcode.com/ppg/ppg_with_haar.php) for face detection
* The JavaScript version of rPPG application was developed by porting the author's Python code using LLM tools (Claude code, Google Gemini, Microsoft Copilot)
* Finding the peak of each heartbeat is done using a machine learning-oriented algorithm called wavelet transformation (based on the Fourier transform)

AI and machine learning (ML) tools model phenomena and predict outcomes using data science. This activity uses AI, ML, and data science to help build understanding of the human heartbeat.

**Data science:** using computation and statistical thinking to tell a story with data.

## Load Libraries We Need

We will use the pandas and NumPy for handling our data, SciPy to process the signals and do some statistics, and MatPlotLib to make our plots.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks, find_peaks_cwt
from scipy.stats import gaussian_kde
from scipy.stats import ttest_ind

## Visualize and Analyze Finger Sensor Pulse Signal

### Load Signal Data from Finger Sensor
This can be either done locally, using the web address provided, or as a Google Drive file. The entire activity can be completed using just the provided data rather than data collected using the micro:bit with a sensor or the webcam rPPG app.

In [ ]:
url1 = "http://thinkingwithcode.com/datascience/sample_pulse_signal_pulse_sensor_amped.csv"
df1 = pd.read_csv(url1)
df1.head(3) # Display the header and the first 3 rows.

In [ ]:
# Convert the signal fit between 0.0 and 1.0
maxsignal1 = df1.loc[df1['signal'].idxmax()]['signal']
df1['signal'] = df1['signal'] / maxsignal1

### Visualize Pulse Data (PPG)
This plot displays the pulse signal over the given time interval which is sometimes called a photoplethysmogram or ppg.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df1['t (ms)'], df1['signal'], color='red', linewidth=2, label='Pulse Signal')
plt.xlabel('t (ms)')
plt.ylabel('signal')
plt.title('Signal vs. Time (Finger)')
plt.grid(True)
plt.legend()
plt.show()

### Detect the Peaks
Here we use an algorithm called [wavelet transformation](https://en.wikipedia.org/wiki/Continuous_wavelet_transform) to look for the peaks in the signal. This is very similar to some machine learning techniques to find signal in a very noisy dataset. The wavelet transformation is related to a [Fourier transform](https://storage.googleapis.com/3blue1brown-website-bucket/lessons/2018/fourier-transforms/DAFC.mp4), which is used to find the various frequencies that make up a signal.

<img src="https://storage.googleapis.com/3blue1brown-website-bucket/lessons/2018/fourier-transforms/DAFC.jpeg" width=640><br>
(Image: <a href="https://www.3blue1brown.com/lessons/fourier-transforms/">3 Blue 1 Brown</a>)

In [ ]:
# Find the peaks using wavelet transformation (ML adjacent tool)
peaks1 = find_peaks_cwt(df1['signal'], widths=np.arange(1, 100))

# Make a list of times corresponding to these peaks (based on index)
peak_times1 = df1['t (ms)'].iloc[peaks1].to_numpy()

# Make a list of the signal values at the peak times
peak_signal1 = df1.loc[df1['t (ms)'].isin(peak_times1), 'signal']

### Plot Signal and Mark the Detected Peaks
This plot shows the original signal but overplots the locations of the peaks discovered using the wavelet transformation algorithm as green dots.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df1['t (ms)'], df1['signal'], color='red', linewidth=2, label='Pulse Signal')
plt.plot(peak_times1, peak_signal1, color='green', marker='o', linestyle='None', label='Peak', markersize=10)
plt.xlabel('ms')
plt.ylabel('signal')
plt.title('Signal vs. Time With Peaks (Finger)')
plt.grid(True)
plt.legend()
plt.show()

### Determine the Interbeat Intervals
* If we want to know the heart rate, we will need to find the time between beats or the interbeat interval (IBI), which varies a bit under most circumstances.

* Since the times were measured in milliseconds, we convert that to seconds before determining the frequency, which is the inverse of the time.

In [ ]:
# Make a list of the differences between beats
ibi1 = np.diff(peak_times1)

# Convert the beats from milliseconds (ms) to beats per minute (bpm)
hr1 = 60000 / ibi1

### Plot Interbeat Intervals
The scatter plot of beats over time shows that the interbeat interval is not a constant for this pulse signal, which is to be expected for a normal heartbeat.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(peak_times1[1:], hr1, s=100, color='blue')
plt.xlabel('t (ms)')
plt.ylabel('Beats Per Minute (BPM)')
plt.title('Heart Rate vs. Time (Finger)')
plt.grid(True)
plt.show()

### Histogram of Heart Rates

Thinking about all the heart rate measurements as a collection or distribution can be instructive. How do the measurements vary? This histogram shows ranges of IBI measurements as bars. The red line shows an approximation of this distribution if we had many, many measurements. Notice how the red line resembles a bell curve, otherwise known as the normal distribution. The red plot is known as a kernel density estimation (KDE).


In [ ]:
plt.figure(figsize=(10, 6))
hist_counts, bin_edges, _ = plt.hist(hr1, bins='auto', alpha=0.7, color='blue', label='Heart Rate Histogram')
# Plot KDE - kind of like the normal distribution
kde = gaussian_kde(hr1)
x_kde = np.linspace(hr1.min(), hr1.max(), 500)

bin_width = bin_edges[1] - bin_edges[0]
plt.plot(x_kde, kde(x_kde) * len(hr1) * bin_width, color='red', linestyle='--', label='Heart Rate KDE')

plt.title('Distribution of Heart Rates (Finger)')
plt.xlabel('Heart Rate (bpm)')
plt.ylabel('Count')
plt.legend()
plt.grid(True)
plt.show()

### Create Box Plot with Descriptive Statistics
One of the things pandas can do is very easily get descriptive statistics like the mean and standard deviation (sd) for a given dataset.

In [ ]:
# Get descriptive statistics (the _ means we didn't need those values)
hrstats1 = pd.Series(hr1).describe()
display(hrstats1)

This box plot shows the distributions of heart rate values while also displaying some of the numbers that describe the dataset.

**How does this visualization of the data differ from the histogram?**

In [ ]:
plt.figure(figsize=(8, 6))
plt.boxplot(hr1)
plt.title('Box Plot of Heart Rate (Finger)')
plt.ylabel('Heart Rate (bpm)')

# Add text annotations for std, min, and max
plt.text(1.1, hrstats1['max'], f'Max: {hrstats1['max']:.2f} BPM', verticalalignment='top', horizontalalignment='left')
plt.text(1.1, hrstats1['min'], f'Min: {hrstats1['min']:.2f} BPM', verticalalignment='bottom', horizontalalignment='left')
plt.text(1.1, hrstats1['mean'], f'Mean: {hrstats1['mean']:.2f} ± {hrstats1['std']:.2f} BPM', verticalalignment='center', horizontalalignment='left')

plt.xticks([]) # Don't display numbers for the horizontal axis.
plt.show()

## Remote PPG (rPPG) from Webcam Signal Analysis

This sample data loaded below was collected using the [remote PPG application](https://thinkingwithcode.com/ppg/) developed by the author. Note that this signal is already displayed on a scale from 0.0 to 1.0. Both the finger and forehead sample were taken at nearly the same time for the same person.

In [ ]:
url2 = "https://thinkingwithcode.com/datascience/sample_pulse_signal_remote_ppg_app.csv"
df2 = pd.read_csv(url2)
df2.head(3)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df2['t (ms)'], df2['signal'], color='green', linewidth=2, label='Pulse Signal (Forehead)')
plt.xlabel('t (ms)')
plt.ylabel('signal')
plt.title('Signal vs. Time (webcam)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Find the peaks using wavelet transformation
peaks2 = find_peaks_cwt(df2['signal'], widths=np.arange(1, 100))

# Get the times corresponding to these peaks (based on index)
peak_times2 = df2['t (ms)'].iloc[peaks2].to_numpy()

# Get the signal values at the peak times
peak_signal2 = df2.loc[df2['t (ms)'].isin(peak_times2), 'signal']

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df2['t (ms)'], df2['signal'], color='green', linewidth=2, label='Pulse Signal (Forehead)')
plt.plot(peak_times2, peak_signal2, color='red', marker='o', linestyle='None', label='Peak', markersize=10)
plt.xlabel('ms')
plt.ylabel('signal')
plt.title('Signal vs. Time With Peaks (Forehead)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Make a list of the differences between beats
ibi2 = np.diff(peak_times2)
# Convert the beats from ms to beats per minute
hr2 = 60000 / ibi2

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(peak_times2[1:], hr2, s=100, color='green')
plt.xlabel('t (ms)')
plt.ylabel('Beats')
plt.title('Heart Rate vs. Time (Webcam)')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
hist_counts, bin_edges, _ = plt.hist(hr2, bins='auto', alpha=0.7, color='green', label='Heart Rate Histogram')

kde = gaussian_kde(hr2)
x_kde = np.linspace(hr2.min(), hr2.max(), 500)
bin_width = bin_edges[1] - bin_edges[0]
plt.plot(x_kde, kde(x_kde) * len(hr2) * bin_width, color='red', linestyle='--', label='Heart Rate KDE')

plt.title('Distribution of Heart Rate (Forehead)')
plt.xlabel('Heart Rate (bpm)')
plt.ylabel('Count')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Get descriptive statistics (the _ means we didn't need those values)
hrstats2 = pd.Series(hr2).describe()
display(hrstats2)

In [ ]:
plt.figure(figsize=(8, 6))
plt.boxplot(hr2)
plt.title('Box Plot of Heart Rate (Forehead)')
plt.ylabel('Heart Rate (bpm)')

# Add text annotations for std, min, and max
plt.text(1.1, hrstats2['max'], f'Max: {hrstats2['max']:.2f} BPM', verticalalignment='top', horizontalalignment='left')
plt.text(1.1, hrstats2['min'], f'Min: {hrstats2['min']:.2f} BPM', verticalalignment='bottom', horizontalalignment='left')
plt.text(1.1, hrstats2['mean'], f'Mean: {hrstats2['mean']:.2f} ± {hrstats2['std']:.2f} BPM', verticalalignment='center', horizontalalignment='left')

plt.xticks([]) # Don't display numbers for the horizontal axis.
plt.show()

## Compare and Contrast Finger Sensor to Webcam Data

In [ ]:
plt.figure(figsize=(10, 6))

# Finger (hr_series1)
hr_series1 = pd.Series(hr1)
hist_counts1, bin_edges1, _ = plt.hist(hr_series1, bins='auto', color="blue", alpha=0.5, label="Finger Histogram")
kde1 = gaussian_kde(hr_series1)
x1_kde = np.linspace(hr_series1.min(), hr_series1.max(), 500)
bin_width1 = bin_edges1[1] - bin_edges1[0]
plt.plot(x1_kde, kde1(x1_kde) * len(hr_series1) * bin_width1, color='blue', linestyle=':')

# Forehead (hr_series2)
hr_series2 = pd.Series(hr2)
hist_counts2, bin_edges2, _ = plt.hist(hr_series2, bins='auto', color="red", alpha=0.1, label="Forehead Histogram")
kde2 = gaussian_kde(hr_series2)
x2_kde = np.linspace(hr_series2.min(), hr_series2.max(), 500)
bin_width2 = bin_edges2[1] - bin_edges2[0]
plt.plot(x2_kde, kde2(x2_kde) * len(hr_series2) * bin_width2, color='red', linestyle=':')

plt.title('Heart Rate Data: Finger & Forehead')
plt.xlabel('Heart Rate (bpm)')
plt.ylabel('Count')
plt.legend()
plt.grid(True)
plt.show()

### Test for Significance
By using a test of significance between the hearbeat data taken from the finger versus the foreheard, we can more definitely state if the two techniques produce comparable results or not for this person.

In [ ]:
# Perform independent t-test
t_stat, p_val = ttest_ind(hr_series1, hr_series2)

print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_val:.3f}")

# Interpret the results
alpha = 0.05
if p_val < alpha:
    print("There is a statistically significant difference between the means of the two heart rate series (reject null hypothesis).")
else:
    print("There is no statistically significant difference between the means of the two heart rate series (fail to reject null hypothesis).")